In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

In [2]:
df=pd.read_csv("messy_ecommerce_sales_data.csv")

In [3]:
df.head()

,ID,Customer_Name,Order_ID,Order_Date,Product,Category,Quantity,Price,Payment_Method,Status,Total
0,100,Customer_100,ORD-41285,2024-11-22,Blender,Home,3,38,Cash on Delivery,Shipped,114.00
1,101,Customer_101,ORD-35783,2025-07-05,Smartphone,Electronics,2,abd,PayPal,Processing,NaN
2,102,Customer_102,ORD-84355,2024-12-23,Tennis Racket,Sports,1,389.05,PayPal,Delivered,389.05
3,103,Customer_103,ORD-57811,2025-03-19,Science,Books,5,233.92,PayPal,Processing,1169.60
4,104,Customer_104,ORD-93614,2025-10-20,Biography,Books,1,552.51,Cash on Delivery,Processing,552.51


In [4]:
df.columns

Index(['ID', ' Customer_Name', 'Order_ID', 'Order_Date', ' Product',
       'Category', 'Quantity', 'Price', 'Payment_Method', 'Status', 'Total'],
      dtype='str')

In [5]:
df.shape

(103, 11)

In [14]:
df.isnull().sum()

ID                  0
 Customer_Name      0
Order_ID            0
Order_Date          3
 Product            0
Category          103
Quantity            6
Price               9
Payment_Method      0
Status              0
Total              14
dtype: int64

In [13]:
df["Quantity"] = pd.to_numeric(df["Quantity"], errors="coerce")
df["Price"] = pd.to_numeric(df["Price"], errors="coerce")

df["Order_Date"] = pd.to_datetime(df["Order_Date"], errors="coerce")
df["Total"] = pd.to_datetime(df["Total"], errors="coerce")

In [15]:
df.isnull().sum()

ID                  0
 Customer_Name      0
Order_ID            0
Order_Date          3
 Product            0
Category          103
Quantity            6
Price               9
Payment_Method      0
Status              0
Total              14
dtype: int64

In [16]:
df["Order_Date"] = df["Order_Date"].fillna(df["Order_Date"].mode()[0])

In [18]:
df["Quantity"] = df["Quantity"].fillna(df["Quantity"].median())

In [19]:
df["Price"] = df["Price"].fillna(df["Price"].median())

In [30]:
df.isnull().sum()

ID                0
 Customer_Name    0
Order_ID          0
Order_Date        0
 Product          0
Quantity          0
Price             0
Payment_Method    0
Status            0
Total             0
dtype: int64

In [29]:
df = df.drop("Category", axis=1)

In [25]:
df["Total"]=df["Total"]=pd.to_numeric(df["Total"])

In [23]:
df['Quantity'] = df['Quantity'].fillna(df['Quantity'].median())
df['Price'] = df['Price'].fillna(df['Price'].median())

In [32]:


df["order_month"] = df["Order_Date"].dt.month
df["order_dayofweek"] = df["Order_Date"].dt.dayofweek

In [33]:
df["Total_Amount"] = df["Quantity"] * df["Price"]

In [34]:
q1 = df["Price"].quantile(0.25)
q3 = df["Price"].quantile(0.75)
iqr = q3 - q1

lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

df["Price"] = df["Price"].clip(lower, upper)

In [35]:
df["Status"] = df["Status"].str.strip().str.title()
df["Status"] = df["Status"].map({"Delivered": 1, "Pending": 0, "Cancelled": 0})

In [37]:
X = df.drop(["Status", "Order_ID",  "Order_Date"], axis=1)
y = df["Status"]

In [38]:
cat_cols = X.select_dtypes(include="object").columns
num_cols = X.select_dtypes(exclude="object").columns

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, num_cols),
    ("cat", categorical_pipeline, cat_cols)
])

C:\Users\dell8\AppData\Local\Temp\ipykernel_11236\321125774.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X.select_dtypes(include="object").columns


In [39]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [40]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

In [43]:
df.columns

Index(['ID', ' Customer_Name', 'Order_ID', 'Order_Date', ' Product',
       'Quantity', 'Price', 'Payment_Method', 'Status', 'Total', 'order_month',
       'order_dayofweek', 'Total_Amount'],
      dtype='str')

In [44]:
df.head()

,ID,Customer_Name,Order_ID,Order_Date,Product,Quantity,Price,Payment_Method,Status,Total,order_month,order_dayofweek,Total_Amount
0,100,Customer_100,ORD-41285,2024-11-22,Blender,3.0,38.000,Cash on Delivery,NaN,114,11,4,114.00
1,101,Customer_101,ORD-35783,2025-07-05,Smartphone,2.0,542.195,PayPal,NaN,-9223372036854775808,7,5,1084.39
2,102,Customer_102,ORD-84355,2024-12-23,Tennis Racket,1.0,389.050,PayPal,1.0,389,12,0,389.05
3,103,Customer_103,ORD-57811,2025-03-19,Science,5.0,233.920,PayPal,NaN,1169,3,2,1169.60
4,104,Customer_104,ORD-93614,2025-10-20,Biography,1.0,552.510,Cash on Delivery,NaN,552,10,0,552.51
